In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def load_and_prepare_data(path):
    """"Loads the Titanic dataset and adds feature engineering
    """
    data = pd.read_csv(path)
    data['FamilySize'] = data['Parch'] + data['SibSp'] + 1
    data['HasCabin'] = data['Cabin'].notnull().astype(int)
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand = False)
    return data

def build_pipeline(numerical_features, categorical_features, int_features):
    """Builds preprocessing pipeline for numeric, integer and categorical data."""
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
    ])

    int_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'median'))
    ])

    full_pipeline = ColumnTransformer([
        ('num', num_pipeline, numerical_features),
        ('int', int_pipeline, int_features),
        ('cat', cat_pipeline, categorical_features)
    ])
    return full_pipeline

def train_classifier(train_x, val_x, train_y, val_y):
    """Find optimal parameters for the Random Forest classifier using GridSearchCV."""
    
    # Define the parameter grid for hyperparameter tuning.
    # Using a limited set of values to reduce training time.
    param_grid = {
        'n_estimators': [100, 150],       # Number of trees in the forest
        'max_depth': [3, 4],              # Maximum depth of each tree
        'min_samples_split': [10, 15],    # Minimum number of samples required to split a node
        'min_samples_leaf': [10, 12], # Minimum number of samples required to be at a leaf node
        'max_features': ['sqrt']          # Number of features to consider when looking for the best split
    }
    
    # Initialize the Random Forest classifier
    model = RandomForestClassifier(random_state = 1)
    
    # GridSearchCV is used instead of nested loops in searching best hyperparameters for random forest classifier
    grid_search = GridSearchCV(model, param_grid, cv = 5,
                               scoring = 'accuracy')
    
    # Fit the models and find the best parameter combination
    grid_search.fit(train_x, train_y)
    
    return grid_search

def train_random_forest(X_processed, Y):
    """Return most accrate model of random tree"""
    train_x, val_x, train_y, val_y = train_test_split(X_processed, Y, random_state = 1)
    grid_search = train_classifier(train_x, val_x, train_y, val_y)
    best_model = grid_search.best_estimator_

    train_acc = evaluate_predictions(best_model, train_x, train_y)
    val_acc = evaluate_predictions(best_model, val_x, val_y)

    print_accuracy(train_acc, val_acc, grid_search)

    return best_model

def evaluate_predictions(model, X, Y):
    """Find accuracy of an algorithm"""
    predictions = model.predict(X)
    return accuracy_score(Y, predictions)
    
def print_accuracy(train_acc, val_acc, grid_search):
    """Print important data for model"""
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Training model accuracy: {train_acc:.4f}")
    print(f"Best cross validation accuracy: {grid_search.best_score_:.4f}")
    print(f"Validation accuracy: {val_acc:.4f}")

def predict_test(test_data, model, pipeline, features):
    """Predict values for test.csv file"""
    X_test = test_data[features]
    X_test_processed = pipeline.transform(X_test)
    return model.predict(X_test_processed)

def main():
    train_path = '/kaggle/input/titanic/train.csv'
    test_path = '/kaggle/input/titanic/test.csv'
    submission_path = 'submission.csv'

    #Separate values into few categories
    numerical_features = ['Age', 'Fare']
    categorical_features = ['Sex', 'Pclass', 'Title', 'HasCabin']
    int_features = ['FamilySize']
    features = numerical_features + categorical_features + int_features

    #load datasets
    train_data = load_and_prepare_data(train_path)
    test_data = load_and_prepare_data(test_path)

    #add pipeline for data preprocessing
    pipeline = build_pipeline(numerical_features, categorical_features, int_features)

    X = train_data[features]
    Y = train_data['Survived']

    #preprocess every variable considered in model
    X_processed = pipeline.fit_transform(X)

    #train model
    model = train_random_forest(X_processed, Y)

    #predict for test file
    test_data['Survived'] = predict_test(test_data, model, pipeline, features)

    submission = test_data[['PassengerId', 'Survived']]

    #save data into submission file
    submission.to_csv(submission_path, index = False)

if __name__ == "__main__":
    main()

Best parameters: {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 150}
Training model accuracy: 0.8278
Best cross validation accuracy: 0.8085
Validation accuracy: 0.8117
